In [ ]:
# ============================================================
# TEMPORAL MENTAL-STATE STUDY — v2 (TIME-SERIES ARCHIVAL)
# Inference-only | All users | Model + Time series saved
# ============================================================

import os, json, pickle, zipfile
import numpy as np
import pandas as pd
from scipy.stats import linregress
from tensorflow.keras.preprocessing.sequence import pad_sequences
import tensorflow as tf

# ============================================================
# CONFIG
# ============================================================

DATA_PATH = "/kaggle/input/sentiment140/training.1600000.processed.noemoticon.csv"
TOKENIZER_PATH = "/kaggle/input/classical-tokenizer/tokenizer.pkl"

MAX_LEN = 50
MIN_POSTS = 50
TAU_DAYS = 7.0
VOL_WINDOW = 20
CHANGE_THRESH = 0.30

BASE_DIR = "temporal_study_v2"
MODEL_DIR = f"{BASE_DIR}/model"
TS_DIR    = f"{BASE_DIR}/timeseries"
SUM_DIR   = f"{BASE_DIR}/summaries"
CP_DIR    = f"{BASE_DIR}/changepoints"
DATA_DIR  = f"{BASE_DIR}/data"

for d in [MODEL_DIR, TS_DIR, SUM_DIR, CP_DIR, DATA_DIR]:
    os.makedirs(d, exist_ok=True)

# ============================================================
# 1. LOAD DATA
# ============================================================

df = pd.read_csv(
    DATA_PATH,
    encoding="latin-1",
    header=None,
    names=["target", "id", "date", "query", "user", "text"],
    usecols=["date", "user", "text"]
)

df["datetime"] = pd.to_datetime(df["date"], errors="coerce", utc=True)
df = df.dropna(subset=["datetime"])
df = df.sort_values(["user", "datetime"]).reset_index(drop=True)

user_counts = df["user"].value_counts()
eligible_users = user_counts[user_counts >= MIN_POSTS].index.tolist()

print(f"[INFO] Eligible users: {len(eligible_users)}")

# ============================================================
# 2. LOAD TOKENIZER
# ============================================================

with open(TOKENIZER_PATH, "rb") as f:
    tokenizer = pickle.load(f)

def encode_text(texts):
    return pad_sequences(
        tokenizer.texts_to_sequences(texts),
        maxlen=MAX_LEN,
        padding="post",
        truncating="post"
    )

# ============================================================
# 3. MODEL INFERENCE (ONCE)
# ============================================================

df["encoded"] = list(encode_text(df["text"].values))
X = np.vstack(df["encoded"].values)

probs = model.predict(X, batch_size=256, verbose=1)
df["sentiment"] = probs[:, 1] - probs[:, 0]

# ============================================================
# 4. TEMPORAL FUNCTIONS
# ============================================================

def decay_weights(times, ref, tau):
    delta = (ref - times).dt.total_seconds() / (60*60*24)
    return np.exp(-delta / tau)

def weighted_trajectory(udf):
    S = []
    for k in range(len(udf)):
        w = decay_weights(
            udf["datetime"].iloc[:k+1],
            udf["datetime"].iloc[k],
            TAU_DAYS
        )
        s = udf["sentiment"].iloc[:k+1].values
        S.append(np.sum(w * s) / np.sum(w))
    return np.array(S)

# ============================================================
# 5. PER-USER COMPUTATION + STORAGE
# ============================================================

rows = []

for user in eligible_users:
    udf = df[df["user"] == user].copy()

    udf["S_weighted"] = weighted_trajectory(udf)
    udf["volatility"] = udf["sentiment"].rolling(
        VOL_WINDOW, min_periods=5
    ).std()

    # Save FULL TIME SERIES (CRITICAL)
    udf[[
        "datetime",
        "sentiment",
        "S_weighted",
        "volatility"
    ]].to_csv(
        f"{TS_DIR}/{user}_timeseries.csv",
        index=False
    )

    # Change-point detection
    delta = udf["sentiment"].rolling(10).mean().diff()
    cps = udf.loc[delta.abs() > CHANGE_THRESH, ["datetime", "sentiment"]]
    cps.to_csv(f"{CP_DIR}/{user}_changepoints.csv", index=False)

    # Drift
    t = np.arange(len(udf))
    slope, _, _, pval, _ = linregress(t, udf["S_weighted"])
    corr = udf["sentiment"].corr(udf["volatility"])

    summary = {
        "user": user,
        "n_posts": len(udf),
        "final_weighted_state": float(udf["S_weighted"].iloc[-1]),
        "mean_weighted_state": float(udf["S_weighted"].mean()),
        "mean_volatility": float(udf["volatility"].mean()),
        "drift_slope": float(slope),
        "drift_p_value": float(pval),
        "sentiment_volatility_corr": float(corr),
        "n_changepoints": int(len(cps))
    }

    with open(f"{SUM_DIR}/{user}_summary.json", "w") as f:
        json.dump(summary, f, indent=2)

    rows.append(summary)

metrics_df = pd.DataFrame(rows)
metrics_df.to_csv(f"{DATA_DIR}/user_metrics.csv", index=False)

# Case study candidates
case_studies = metrics_df[
    (metrics_df["n_changepoints"] > 0) |
    (metrics_df["mean_volatility"] > metrics_df["mean_volatility"].quantile(0.90)) |
    (metrics_df["drift_slope"].abs() > metrics_df["drift_slope"].abs().quantile(0.90))
]

case_studies.to_csv(f"{DATA_DIR}/case_study_candidates.csv", index=False)

# ============================================================
# 6. SAVE MODEL + TOKENIZER
# ============================================================

model.save(f"{MODEL_DIR}/sentiment_model.keras")

with open(f"{MODEL_DIR}/tokenizer.pkl", "wb") as f:
    pickle.dump(tokenizer, f)

# ============================================================
# 7. README
# ============================================================

with open(f"{BASE_DIR}/README.txt", "w") as f:
    f.write(
        "Temporal Mental-State Analysis v2\n"
        "--------------------------------\n"
        "Inference-only sentiment analysis on Sentiment140.\n"
        "Per-user time series saved to enable post-hoc visualization\n"
        "without re-running the model.\n"
    )

# ============================================================
# 8. ZIP EVERYTHING
# ============================================================

zip_name = "temporal_study_v2_timeseries_bundle.zip"

with zipfile.ZipFile(zip_name, "w", zipfile.ZIP_DEFLATED) as zipf:
    for folder, _, files in os.walk(BASE_DIR):
        for file in files:
            full_path = os.path.join(folder, file)
            zipf.write(
                full_path,
                arcname=full_path.replace(BASE_DIR + "/", "")
            )

print("✔ Temporal study v2 complete.")
print(f"📦 ZIP saved as: {zip_name}")
